# Enum — Nível Intermediário

**Objetivo:** escolher o tipo certo da família `enum`, dar comportamento aos membros, aceitar entradas tolerantes e
integrar Enums com configuração, JSON, dataclasses e Pydantic.

Roteiro:

1. `StrEnum` e `IntEnum`: quando o membro precisa "ser" o valor
2. `str()`, `format()` e f-strings: a diferença que quebra logs e SQL
3. `Flag`: combinando opções (e a pegadinha do alias de combinação)
4. `IntFlag`
5. Métodos, propriedades e `classmethod` no Enum
6. Membros ricos: valor + atributos com `__init__`
7. `_missing_`: entrada tolerante (case-insensitive, sinônimos)
8. `_generate_next_value_` e `@verify`
9. Enum em **configuração** (env vars, YAML)
10. Enum em **JSON**, **dataclass** e **Pydantic** (DTOs de API + JSON Schema)

In [1]:
import json, os
from enum import Enum, IntEnum, StrEnum, Flag, IntFlag, auto, unique, verify, UNIQUE, CONTINUOUS, NAMED_FLAGS

## 1. `StrEnum` e `IntEnum`

`StrEnum` (3.11+) é **subclasse de `str`**: o membro é uma string de verdade. `IntEnum` é subclasse de `int`.
Isso permite comparar direto com o valor bruto e passar o membro para qualquer função que espera `str`/`int`.

In [2]:
class Environment(StrEnum):
    DEV = auto()          # em StrEnum, auto() gera o nome em minúsculas
    STAGING = auto()
    PROD = auto()

class HttpStatus(IntEnum):
    OK = 200
    CREATED = 201
    BAD_REQUEST = 400
    NOT_FOUND = 404

print(f"list(Environment): {list(Environment)}")
print(f'Environment.PROD == "prod": {Environment.PROD == "prod"}')
print(f"isinstance(Environment.PROD, str): {isinstance(Environment.PROD, str)}")
print(f"Environment.PROD.upper(): {Environment.PROD.upper()}")
print(f"HttpStatus.NOT_FOUND == 404: {HttpStatus.NOT_FOUND == 404}")
print(f"HttpStatus.NOT_FOUND + 1: {HttpStatus.NOT_FOUND + 1}")
print(f"HttpStatus(201).name: {HttpStatus(201).name}")
print(f"sorted([HttpStatus.NOT_FOUND, HttpStatus.OK, HttpStatus.BAD_REQUEST]): {sorted([HttpStatus.NOT_FOUND, HttpStatus.OK, HttpStatus.BAD_REQUEST])}")

list(Environment): [<Environment.DEV: 'dev'>, <Environment.STAGING: 'staging'>, <Environment.PROD: 'prod'>]
Environment.PROD == "prod": True
isinstance(Environment.PROD, str): True
Environment.PROD.upper(): PROD
HttpStatus.NOT_FOUND == 404: True
HttpStatus.NOT_FOUND + 1: 405
HttpStatus(201).name: CREATED
sorted([HttpStatus.NOT_FOUND, HttpStatus.OK, HttpStatus.BAD_REQUEST]): [<HttpStatus.OK: 200>, <HttpStatus.BAD_REQUEST: 400>, <HttpStatus.NOT_FOUND: 404>]


O preço: você perde parte da segurança. Com `IntEnum`, `HttpStatus.OK == 200 == Priority.X` (se algum outro IntEnum valer 200)
— enums diferentes passam a ser "iguais" via o inteiro. Use `IntEnum` só quando precisar interoperar com APIs numéricas.

In [3]:
class Weekday(IntEnum):
    MONDAY = 1

class Level(IntEnum):
    LOW = 1

print(f"enums diferentes iguais via int: {Weekday.MONDAY == Level.LOW}")

enums diferentes iguais via int: True


## 2. `str()`, `format()` e f-strings

Diferença sutil que já quebrou muita query SQL e chave de cache:

In [4]:
class PlainColor(Enum):
    RED = "red"

class StrColor(StrEnum):
    RED = "red"

for m in (PlainColor.RED, StrColor.RED, HttpStatus.OK):
    print(f"{type(m).__name__:<11} str()={str(m)!r:<18} f-string={f'{m}'!r:<18} .value={m.value!r}")

PlainColor  str()='PlainColor.RED'   f-string='PlainColor.RED'   .value='red'
StrColor    str()='red'              f-string='red'              .value='red'
HttpStatus  str()='200'              f-string='200'              .value=200


Regra prática: quando o destino é texto externo (SQL, URL, chave de cache, JSON), use **`.value` explicitamente** —
funciona igual para todos os tipos e deixa a intenção clara.

In [5]:
# Errado: depende de str()/format() do tipo de Enum
cache_key_wrong = f"products:color={PlainColor.RED}"
sql_wrong = f"SELECT * FROM cars WHERE color = '{PlainColor.RED}'"
print(f"cache_key_wrong: {cache_key_wrong}")
print(f"sql_wrong: {sql_wrong}")

# Correto: .value explícito (e parâmetro na query, nunca interpolação)
cache_key = f"products:color={PlainColor.RED.value}"
sql, params = "SELECT * FROM cars WHERE color = ?", (PlainColor.RED.value,)
print(f"cache_key: {cache_key}")
print(f"sql, params: {sql}, {params}")
print(f"mesma chave para Enum e StrEnum: {f'color={PlainColor.RED.value}' == f'color={StrColor.RED.value}'}")

cache_key_wrong: products:color=PlainColor.RED
sql_wrong: SELECT * FROM cars WHERE color = 'PlainColor.RED'
cache_key: products:color=red
sql, params: SELECT * FROM cars WHERE color = ?, ('red',)
mesma chave para Enum e StrEnum: True


## 3. `Flag`: combinando opções

`Enum` representa **um** estado por vez. `Flag` representa um **conjunto** de opções ativas, usando bits:
cada membro é uma potência de 2 e a combinação é feita com `|`.

| operação | significado |
|---|---|
| `a \| b` | união (ativa ambos) |
| `a & b` | interseção |
| `x in combo` | o bit `x` está ativo? |
| `~a` | complemento (todos os outros) |
| `combo ^ a` | alterna `a` |
| `Flag(0)` | nenhum ativo (falsy) |

In [6]:
class Permission(Flag):
    READ = auto()        # 1
    WRITE = auto()       # 2
    DELETE = auto()      # 4
    SHARE = auto()       # 8

    EDITOR = READ | WRITE                  # combinação nomeada (alias)
    ADMIN = READ | WRITE | DELETE | SHARE

teacher = Permission.READ | Permission.WRITE | Permission.DELETE
student = Permission.READ

print(f"teacher: {teacher}")
print(f"valor: {teacher.value}")
print(f"DELETE in teacher: {Permission.DELETE in teacher}")
print(f"WRITE in student: {Permission.WRITE in student}")
print(f"student tem tudo de EDITOR?: {Permission.EDITOR in student}")
print(f"teacher sem DELETE: {teacher & ~Permission.DELETE}")
print(f"membros individuais de teacher: {list(teacher)}")
print(f"iterando a classe: {list(Permission)}  <- combinações nomeadas não aparecem")
print(f"nenhuma permissão: {Permission(0)}")
print(f"bool(Permission(0)): {bool(Permission(0))}")

teacher: Permission.READ|WRITE|DELETE
valor: 7
DELETE in teacher: True
WRITE in student: False
student tem tudo de EDITOR?: False
teacher sem DELETE: Permission.EDITOR
membros individuais de teacher: [<Permission.READ: 1>, <Permission.WRITE: 2>, <Permission.DELETE: 4>]
iterando a classe: [<Permission.READ: 1>, <Permission.WRITE: 2>, <Permission.DELETE: 4>, <Permission.SHARE: 8>]  <- combinações nomeadas não aparecem
nenhuma permissão: Permission(0)
bool(Permission(0)): False


### A pegadinha do alias de combinação

Se um membro de `Flag` recebe um valor que **não é potência de 2**, ele não é uma opção nova: é um **apelido da combinação**
dos bits correspondentes. Veja o que acontece quando alguém define `BLACK = 12` achando que criou uma cor nova:

In [7]:
class Color(Flag):
    RED = 1
    GREEN = 2
    BLUE = 4
    YELLOW = 8
    BLACK = 12         # 12 = 8 + 4 -> é YELLOW | BLUE !

combo = Color.YELLOW | Color.BLUE
print(f"YELLOW | BLUE: {combo}")                  # imprime Color.BLACK
print(f"BLACK é a combinação?: {Color.BLACK is combo}")
print(f"BLACK aparece na iteração?: {list(Color)}")
print(f"bits de BLACK: {list(Color.BLACK)}")

YELLOW | BLUE: Color.BLACK
BLACK é a combinação?: True
BLACK aparece na iteração?: [<Color.RED: 1>, <Color.GREEN: 2>, <Color.BLUE: 4>, <Color.YELLOW: 8>]
bits de BLACK: [<Color.BLUE: 4>, <Color.YELLOW: 8>]


⚠️ **Atenção: os validadores do `enum` não pegam esse erro.** Como `YELLOW` e `BLUE` têm nome, a combinação `12` é considerada
um alias legítimo — `@verify(NAMED_FLAGS)`, `@verify(UNIQUE)` e `@unique` aceitam a classe sem reclamar:

In [8]:
from enum import verify, unique, NAMED_FLAGS, UNIQUE

for name, decorator in [("@verify(NAMED_FLAGS)", verify(NAMED_FLAGS)), ("@verify(UNIQUE)", verify(UNIQUE)), ("@unique", unique)]:
    try:
        @decorator
        class ColorChecked(Flag):
            RED = 1
            GREEN = 2
            BLUE = 4
            YELLOW = 8
            BLACK = 12
        print(f"{name} aceitou BLACK = 12: {ColorChecked.BLACK!r}")
    except ValueError as e:
        print(f"{name}: {type(e).__name__}: {e}")

@verify(NAMED_FLAGS) aceitou BLACK = 12: <ColorChecked.BLACK: 12>
@verify(UNIQUE) aceitou BLACK = 12: <ColorChecked.BLACK: 12>
@unique aceitou BLACK = 12: <ColorChecked.BLACK: 12>


`NAMED_FLAGS` só reclama quando a combinação usa um bit **sem nome** (ex.: `BLACK = 12` sem `YELLOW`/`BLUE` definidos).
A correção depende da **intenção** — e cada uma tem um código diferente:

**Correção A — `BLACK` é uma cor independente:** use `auto()` para todas as opções. Cada membro ganha o próprio bit (`BLACK` vira 16)
e nenhum número escrito à mão pode colidir.

In [9]:
class Color(Flag):
    RED = auto()       # 1
    GREEN = auto()     # 2
    BLUE = auto()      # 4
    YELLOW = auto()    # 8
    BLACK = auto()     # 16 -> bit próprio

combo = Color.YELLOW | Color.BLUE
print(f"Color.BLACK.value: {Color.BLACK.value}")
print(f"Color.YELLOW | Color.BLUE: {combo}")
print(f"Color.BLACK is combo: {Color.BLACK is combo}")
print(f"list(Color): {list(Color)}")
print(f"Color.BLACK in combo: {Color.BLACK in combo}")

Color.BLACK.value: 16
Color.YELLOW | Color.BLUE: Color.BLUE|YELLOW
Color.BLACK is combo: False
list(Color): [<Color.RED: 1>, <Color.GREEN: 2>, <Color.BLUE: 4>, <Color.YELLOW: 8>, <Color.BLACK: 16>]
Color.BLACK in combo: False


**Correção B — `BLACK` é mesmo a combinação de `YELLOW` e `BLUE`:** escreva a combinação **com os nomes**, não com o número.
O comportamento é o mesmo do `BLACK = 12`, mas a intenção fica explícita no código e não quebra se os bits mudarem.
Junto, um teste que documenta quais aliases de combinação são esperados — se alguém adicionar um `12` por engano, o teste falha.

In [10]:
class Color(Flag):
    RED = auto()
    GREEN = auto()
    BLUE = auto()
    YELLOW = auto()
    BLACK = YELLOW | BLUE      # combinação intencional, escrita com nomes

def composite_members(flag_cls: type[Flag]) -> set[str]:
    """Membros nomeados que são combinação de vários bits (aliases de combinação)."""
    return {name for name, m in flag_cls.__members__.items() if m.value and m.value & (m.value - 1)}

print(f"Color.BLACK: {Color.BLACK!r}")
print(f"Color.YELLOW | Color.BLUE: {Color.YELLOW | Color.BLUE}")
print(f"list(Color): {list(Color)}")
print(f"composite_members(Color): {composite_members(Color)}")

assert composite_members(Color) == {"BLACK"}, "alias de combinação inesperado no Flag Color"
print("teste de aliases esperados: ok")

Color.BLACK: <Color.BLACK: 12>
Color.YELLOW | Color.BLUE: Color.BLACK
list(Color): [<Color.RED: 1>, <Color.GREEN: 2>, <Color.BLUE: 4>, <Color.YELLOW: 8>]
composite_members(Color): {'BLACK'}
teste de aliases esperados: ok


## 4. `IntFlag`

Como `Flag`, mas também é `int`: interopera com bitmasks numéricas (permissões de arquivo, flags de APIs em C, colunas inteiras no banco).

In [11]:
class FileMode(IntFlag):
    X = 1
    W = 2
    R = 4

mode = FileMode.R | FileMode.W
print(f"mode: {mode}")
print(f"int(mode): {int(mode)}")
print(f"oct(FileMode.R | FileMode.W | FileMode.X): {oct(FileMode.R | FileMode.W | FileMode.X)}")
print(f"aceita int direto: {FileMode(5)}")
print(f"combina com int: {mode | 1}")

mode: 6
int(mode): 6
oct(FileMode.R | FileMode.W | FileMode.X): 0o7
aceita int direto: 5
combina com int: 7


## 5. Métodos, propriedades e `classmethod`

Um Enum é uma classe: pode ter métodos. Isso mantém **regras ligadas ao vocabulário** perto dele, em vez de espalhadas em `if`s.

In [12]:
class OrderStatus(StrEnum):
    PENDING = auto()
    PAID = auto()
    SHIPPED = auto()
    DELIVERED = auto()
    CANCELLED = auto()

    @property
    def is_final(self) -> bool:
        return self in {OrderStatus.DELIVERED, OrderStatus.CANCELLED}

    @property
    def label_pt(self) -> str:
        return {"pending": "Pendente", "paid": "Pago", "shipped": "Enviado",
                "delivered": "Entregue", "cancelled": "Cancelado"}[self.value]

    def can_cancel(self) -> bool:
        return self in {OrderStatus.PENDING, OrderStatus.PAID}

    @classmethod
    def open_statuses(cls) -> list["OrderStatus"]:
        return [s for s in cls if not s.is_final]

print(f"OrderStatus.PAID.label_pt: {OrderStatus.PAID.label_pt}")
print(f"OrderStatus.PAID.can_cancel(): {OrderStatus.PAID.can_cancel()}")
print(f"OrderStatus.SHIPPED.can_cancel(): {OrderStatus.SHIPPED.can_cancel()}")
print(f"abertos: {OrderStatus.open_statuses()}")
print(f"opções para um <select>: {[(s.value, s.label_pt) for s in OrderStatus]}")

OrderStatus.PAID.label_pt: Pago
OrderStatus.PAID.can_cancel(): True
OrderStatus.SHIPPED.can_cancel(): False
abertos: [<OrderStatus.PENDING: 'pending'>, <OrderStatus.PAID: 'paid'>, <OrderStatus.SHIPPED: 'shipped'>]
opções para um <select>: [('pending', 'Pendente'), ('paid', 'Pago'), ('shipped', 'Enviado'), ('delivered', 'Entregue'), ('cancelled', 'Cancelado')]


## 6. Membros ricos: valor + atributos

Quando cada membro precisa de vários dados (código, descrição, limite, custo...), use uma tupla como valor e um `__init__`
para dar nome aos pedaços. O `value` continua sendo a tupla, a menos que você defina `_value_` no `__new__`.

In [13]:
class Plan(Enum):
    #        (código,  preço mensal, max usuários, suporte)
    FREE = ("free", 0, 1, "comunidade")
    PRO = ("pro", 49, 5, "email")
    TEAM = ("team", 199, 50, "chat")

    def __init__(self, code: str, monthly_price: int, max_users: int, support: str):
        self.code = code
        self.monthly_price = monthly_price
        self.max_users = max_users
        self.support = support

    def fits(self, users: int) -> bool:
        return users <= self.max_users

print(f"Plan.PRO.code: {Plan.PRO.code}")
print(f"Plan.PRO.monthly_price: {Plan.PRO.monthly_price}")
print(f"Plan.PRO.support: {Plan.PRO.support}")
print(f"menor plano para 12 usuários: {next(p for p in Plan if p.fits(12))}")
print(f"value é a tupla: {Plan.PRO.value}")

Plan.PRO.code: pro
Plan.PRO.monthly_price: 49
Plan.PRO.support: email
menor plano para 12 usuários: Plan.TEAM
value é a tupla: ('pro', 49, 5, 'email')


Para que o **valor** seja só o código (e `Plan("pro")` funcione), defina `__new__` e atribua `_value_`:

In [14]:
class Region(Enum):
    def __new__(cls, code: str, latency_ms: int, gdpr: bool):
        obj = object.__new__(cls)
        obj._value_ = code                  # o valor "oficial" do membro
        obj.latency_ms = latency_ms
        obj.gdpr = gdpr
        return obj

    SA_EAST = ("sa-east-1", 20, False)
    US_EAST = ("us-east-1", 120, False)
    EU_WEST = ("eu-west-1", 190, True)

print(f'Region("eu-west-1"): {Region("eu-west-1")}')
print(f"Region.EU_WEST.value: {Region.EU_WEST.value}")
print(f"Region.EU_WEST.gdpr: {Region.EU_WEST.gdpr}")
print(f"mais próxima: {min(Region, key=lambda r: r.latency_ms)}")

Region("eu-west-1"): Region.EU_WEST
Region.EU_WEST.value: eu-west-1
Region.EU_WEST.gdpr: True
mais próxima: Region.SA_EAST


## 7. `_missing_`: entrada tolerante

`_missing_` é chamado quando `Enum(valor)` não encontra o valor. Devolva um membro para aceitar a entrada, ou `None` para manter o erro.
Ideal para **normalizar** variações vindas de humanos e sistemas legados — num lugar só.

In [15]:
class LogLevel(StrEnum):
    DEBUG = "debug"
    INFO = "info"
    WARNING = "warning"
    ERROR = "error"

    @classmethod
    def _missing_(cls, value):
        if isinstance(value, str):
            v = value.strip().lower()
            synonyms = {"warn": "warning", "err": "error", "information": "info"}
            v = synonyms.get(v, v)
            for member in cls:
                if member.value == v:
                    return member
        return None

for raw in ["INFO", " Warn ", "err", "debug"]:
    print(f"{raw!r:<10} -> {LogLevel(raw)!r}")
try:
    LogLevel("verbose")
except ValueError as e:
    print(f"ValueError: {e}")

'INFO'     -> <LogLevel.INFO: 'info'>
' Warn '   -> <LogLevel.WARNING: 'warning'>
'err'      -> <LogLevel.ERROR: 'error'>
'debug'    -> <LogLevel.DEBUG: 'debug'>
ValueError: 'verbose' is not a valid LogLevel


## 8. `_generate_next_value_` e `@verify`

### Controlando o `auto()`
`_generate_next_value_` define o que `auto()` produz. Precisa ser declarado **antes** dos membros.

In [16]:
class Event(StrEnum):
    @staticmethod
    def _generate_next_value_(name, start, count, last_values):
        return name.lower().replace("__", ".")      # ORDER__PAID -> order.paid

    ORDER__CREATED = auto()
    ORDER__PAID = auto()
    CUSTOMER__CREATED = auto()

print(f"list(Event): {list(Event)}")

list(Event): [<Event.ORDER__CREATED: 'order.created'>, <Event.ORDER__PAID: 'order.paid'>, <Event.CUSTOMER__CREATED: 'customer.created'>]


### Validando a definição com `@verify` (3.11+)

| verificação | garante |
|---|---|
| `UNIQUE` | nenhum alias (como `@unique`) |
| `CONTINUOUS` | valores inteiros sem buracos (útil para códigos sequenciais) |
| `NAMED_FLAGS` | todo bit usado em combinações de `Flag` tem nome próprio |

In [17]:
for name, build in {
    "CONTINUOUS": lambda: verify(CONTINUOUS)(Enum("Step", {"ONE": 1, "TWO": 2, "FOUR": 4})),
    "NAMED_FLAGS": lambda: verify(NAMED_FLAGS)(Flag("Opt", {"A": 1, "B": 2, "AB_AND_C": 7})),
    "UNIQUE": lambda: verify(UNIQUE)(Enum("Tier", {"GOLD": 1, "OURO": 1})),
}.items():
    try:
        build()
    except ValueError as e:
        print(f"{name:<12} ValueError: {e}")

CONTINUOUS   ValueError: invalid enum 'Step': missing values 3
NAMED_FLAGS  ValueError: invalid Flag 'Opt': alias AB_AND_C is missing value 0x4 [use enum.show_flag_values(value) for details]
UNIQUE       ValueError: aliases found in <enum 'Tier'>: OURO -> GOLD


## 9. Enum em configuração

Variáveis de ambiente e arquivos de config são texto. Converter para Enum **no carregamento** garante que o resto do código
só lida com opções válidas — e o erro aparece no startup, listando as opções.

In [18]:
class CacheBackend(StrEnum):
    MEMORY = auto()
    REDIS = auto()
    DISABLED = auto()

class Feature(Flag):
    SIGNUP = auto()
    EXPORT = auto()
    BETA_UI = auto()
    AI_ASSISTANT = auto()

    @classmethod
    def parse(cls, raw: str) -> "Feature":
        """'signup, export' -> Feature.SIGNUP | Feature.EXPORT"""
        result = cls(0)
        for name in filter(None, (p.strip().upper() for p in raw.split(","))):
            try:
                result |= cls[name]
            except KeyError:
                raise ValueError(f"feature {name!r} desconhecida. Opções: {[m.name.lower() for m in cls]}") from None
        return result

def enum_from_env(var: str, enum_cls: type[Enum], default: Enum) -> Enum:
    raw = os.environ.get(var)
    if raw is None:
        return default
    try:
        return enum_cls(raw)
    except ValueError:
        options = ", ".join(str(m.value) for m in enum_cls)
        raise SystemExit(f"config inválida: {var}={raw!r}. Opções: {options}") from None

os.environ.update({"APP_ENV": "PROD", "APP_LOG_LEVEL": "warn", "APP_CACHE": "redis", "APP_FEATURES": "signup, ai_assistant"})

class Env(StrEnum):
    DEV = auto(); STAGING = auto(); PROD = auto()
    @classmethod
    def _missing_(cls, value):
        return cls.__members__.get(str(value).upper())

config = {
    "env": enum_from_env("APP_ENV", Env, Env.DEV),
    "log_level": enum_from_env("APP_LOG_LEVEL", LogLevel, LogLevel.INFO),
    "cache": enum_from_env("APP_CACHE", CacheBackend, CacheBackend.MEMORY),
    "features": Feature.parse(os.environ["APP_FEATURES"]),
}
print(f"config: {config}")
print(f'assistente ligado?: {Feature.AI_ASSISTANT in config["features"]}')

os.environ["APP_CACHE"] = "memcached"
try:
    enum_from_env("APP_CACHE", CacheBackend, CacheBackend.MEMORY)
except SystemExit as e:
    print(f"{type(e).__name__}: {e}")
for k in ("APP_ENV", "APP_LOG_LEVEL", "APP_CACHE", "APP_FEATURES"):
    os.environ.pop(k)

config: {'env': <Env.PROD: 'prod'>, 'log_level': <LogLevel.WARNING: 'warning'>, 'cache': <CacheBackend.REDIS: 'redis'>, 'features': <Feature.SIGNUP|AI_ASSISTANT: 9>}
assistente ligado?: True
SystemExit: config inválida: APP_CACHE='memcached'. Opções: memory, redis, disabled


Com `pydantic-settings` (veja `pydantic/02`) isso é automático: basta anotar o campo com o Enum.

## 10. Enum em JSON, dataclass e Pydantic

### JSON puro
`json.dumps` serializa `StrEnum`/`IntEnum` (são `str`/`int`), mas **não** `Enum` puro nem `Flag`:

In [19]:
print(f'json.dumps({{"env": Env.PROD, "status": HttpStatus.OK}}): {json.dumps({"env": Env.PROD, "status": HttpStatus.OK})}')
try:
    json.dumps({"color": PlainColor.RED})
except TypeError as e:
    print(f"TypeError: {e}")
enum_to_value = lambda o: o.value if isinstance(o, Enum) else str(o)
print(f'json.dumps({{"color": PlainColor.RED}}, default=enum_to_value): {json.dumps({"color": PlainColor.RED}, default=enum_to_value)}')

json.dumps({"env": Env.PROD, "status": HttpStatus.OK}): {"env": "prod", "status": 200}
TypeError: Object of type PlainColor is not JSON serializable
json.dumps({"color": PlainColor.RED}, default=enum_to_value): {"color": "red"}


### Dataclass
Dataclass não converte: se o dado vem de fora, converta no `__post_init__` (ou na borda).

In [20]:
from dataclasses import dataclass, asdict

@dataclass
class Order:
    id: int
    status: OrderStatus

    def __post_init__(self):
        self.status = OrderStatus(self.status)       # aceita "paid" ou OrderStatus.PAID

o = Order(1, "paid")
print(f"o: {o}")
print(f"is_final: {o.status.is_final}")
print(f"{asdict(o)} -> {json.dumps(asdict(o))}")

o: Order(id=1, status=<OrderStatus.PAID: 'paid'>)
is_final: False
{'id': 1, 'status': <OrderStatus.PAID: 'paid'>} -> {"id": 1, "status": "paid"}


### Pydantic: validação, mensagens e JSON Schema de graça
Anotar o campo com o Enum faz o Pydantic: converter o valor, rejeitar valores fora do conjunto listando as opções e
publicar `"enum": [...]` no JSON Schema (OpenAPI). `use_enum_values=True` guarda o `.value` em vez do membro.

In [21]:
from pydantic import BaseModel, ConfigDict, ValidationError

class OrderOut(BaseModel):
    id: int
    status: OrderStatus
    last_http_status: HttpStatus | None = None

class OrderFlat(BaseModel):
    model_config = ConfigDict(use_enum_values=True)
    id: int
    status: OrderStatus

m = OrderOut.model_validate_json('{"id": 1, "status": "shipped"}')
print(f"repr(m.status): {repr(m.status)}")
print(f"JSON: {m.model_dump_json()}")
print(f'use_enum_values: {repr(OrderFlat(id=1, status="paid").status)}')

try:
    OrderOut(id=1, status="lost")
except ValidationError as e:
    print(f'e.errors(include_url=False)[0]["msg"]: {e.errors(include_url=False)[0]["msg"]}')

print(f'json.dumps(OrderOut.model_json_schema()["$defs"]["OrderStatus"]): {json.dumps(OrderOut.model_json_schema()["$defs"]["OrderStatus"])}')

repr(m.status): <OrderStatus.SHIPPED: 'shipped'>
JSON: {"id":1,"status":"shipped","last_http_status":null}
use_enum_values: 'paid'
e.errors(include_url=False)[0]["msg"]: Input should be 'pending', 'paid', 'shipped', 'delivered' or 'cancelled'
json.dumps(OrderOut.model_json_schema()["$defs"]["OrderStatus"]): {"enum": ["pending", "paid", "shipped", "delivered", "cancelled"], "title": "OrderStatus", "type": "string"}


## Resumo

| precisa de... | use |
|---|---|
| vocabulário seguro, sem confundir com valor bruto | `Enum` |
| membro que **é** string (JSON, headers, chaves) | `StrEnum` |
| interoperar com códigos numéricos | `IntEnum` |
| várias opções ao mesmo tempo | `Flag` com `auto()`; combinações escritas com nomes (`BLACK = YELLOW \| BLUE`) + teste de aliases |
| bitmask numérica | `IntFlag` |
| regras ligadas ao vocabulário | métodos / `@property` no Enum |
| vários dados por membro | `__init__` (ou `__new__` + `_value_`) |
| entrada tolerante | `_missing_` |
| texto externo (SQL, cache, URL) | sempre `.value` |
| validação + schema | campo Pydantic anotado com o Enum |

### Exercícios
1. Crie `Currency(StrEnum)` com membros ricos (`symbol`, `decimals`) e um método `format(amount)`.
2. Faça `Permission.parse("read,write")` aceitar também `"*"` = todas.
3. Escreva `_missing_` para `HttpStatus` que aceite `"404"` (string) além de `404`.

➡️ Próximo: [`03_enum_avancado.ipynb`](03_enum_avancado.ipynb)